# mini-vLLM â€” GPU Correctness & Benchmark Notebook

**Runtime:** GPU T4 â€” Kaggle: Settings â†’ Accelerator â†’ GPU T4 x1

In [ ]:
# ── Clone / update ──────────────────────────────────────────────────────────
import os
from pathlib import Path

BRANCH    = 'milestone/m1-correctness-baseline'
CLONE_URL = 'https://github.com/shlokkvaishnav/LLM-Inference-Engine.git'
# Absolute path prevents os.chdir from nesting deeper on each re-run.
LOCAL_DIR = Path('/kaggle/working/mini-vllm')

if LOCAL_DIR.exists():
    !git -C {LOCAL_DIR} pull
else:
    !git clone --branch {BRANCH} {CLONE_URL} {LOCAL_DIR}

os.chdir(LOCAL_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Register our package on sys.path without touching any pre-installed packages.
!pip install -e . --no-deps -q
print('Install complete.')

In [ ]:
# transformers==4.46.3 version-checks these packages at import time
# (dependency_versions_check.py). Two have upper bounds that Kaggle violates:
#   tokenizers:      needs >=0.20,<0.21  — Kaggle has 0.22.x
#   huggingface-hub: needs >=0.23.2,<1.0 — Kaggle has 1.x
# Everything else (safetensors, accelerate, tqdm, numpy, …) only has a lower
# bound so Kaggle's newer versions are fine.
#
# Strategy: install exact pins into an isolated --target dir and prepend it
# to sys.path. --upgrade avoids the "directory already exists" warning on
# re-runs. Evict sys.modules so a prior failed import cannot leave stale
# cached entries that bypass our path.
import subprocess, sys, os

TARGET = "/kaggle/working/pkgs"
os.makedirs(TARGET, exist_ok=True)

for pkg in [
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "huggingface_hub==0.26.2",
]:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        pkg, "--no-deps", "-q", "--target", TARGET, "--upgrade",
    ])

# Evict stale cached modules before the next import.
_prefixes = ("transformers", "tokenizers", "huggingface_hub")
for _k in [k for k in sys.modules
           if any(k == p or k.startswith(p + ".") for p in _prefixes)]:
    del sys.modules[_k]

if TARGET not in sys.path:
    sys.path.insert(0, TARGET)

print("pinned: transformers==4.46.3  tokenizers==0.20.3  huggingface_hub==0.26.2")

In [ ]:
import torch, transformers
print(f'PyTorch      : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# PYTHONPATH ensures the pytest subprocess (a fresh Python process that does
# not inherit the notebook kernel's sys.path edits) also finds our pinned
# transformers/tokenizers/huggingface_hub before the system site-packages.
!MINI_VLLM_TEST_MODEL='TinyLlama/TinyLlama-1.1B-Chat-v1.0' \n MINI_VLLM_TEST_DEVICE='cuda' \n PYTHONPATH=/kaggle/working/pkgs \n python -m pytest tests/test_correctness.py -v -s 2>&1

In [ ]:
!MINI_VLLM_TEST_MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0 MINI_VLLM_TEST_DEVICE=cuda PYTHONPATH=/kaggle/working/pkgs python -m pytest tests/test_correctness.py -v -s 2>&1

---
## M3 — Continuous batching ✓ (Milestone 3 landed)

`test_continuous_batch_matches_single`: 4 prompts through LLMEngine with max_batch_size=2.
The engine admits 2, then as each finishes it admits the next one.
Each sequence's output must match the HuggingFace greedy baseline token-for-token.

Key design points for interviews:
- `step()` runs **every decode step** (tight loop), not once per batch
- `decode_one()` is O(N) passes/step — deliberate simplicity; M4 fixes this with paged attention
- `Scheduler.free()` immediately opens a slot; next `step()` admits a waiter

## M4 — Paged KV-cache *(added when milestone lands)*
## M5 — Quantization *(added when milestone lands)*
## M7 — Benchmark suite + P50/P95/P99 load test *(added when milestone lands)*